# Track B — 씬파일러 사후 검증, 최종 튜닝 모델 기준 재수행 (track_b_12)

담당: 김태헌 | 선행 노트북: track_b_07·08(튜닝된 XGBoost 확정), track_b_11(서윤 — 이탈 코호트 발굴)

## 이 노트북의 목적

track_b_11 전체를 다시 돌리지 않고, **서윤님이 전달한 두 파일만으로 독립적으로 실행**되도록
구성했다.

- `track_b_exit_cohort.csv` — 정제된 이탈 코호트 2,931명(CUST_ID, y_true만 있음, 양성 39명)
- `통신카드CB_씬파일러.csv` — 씬파일러 44,110명을 202103~202212 분기별로 추적한 종단 데이터
  (신용이력 보유군은 포함 안 됨, 739개 컬럼 = CUST_ID/BASE_YM + 원본 후보변수 738개)

이탈 코호트의 202103(씬파일러 시절) 피처 값을 이 파일에서 직접 뽑아서, track_b_07/08에서
튜닝된 XGBoost 모델(신용이력 보유군 285,890명으로 학습)에 넣어 점수를 매기고, 실제 연체
정답과 대조한다.

## ⚠️ WOE 인코딩 변수 4개는 근사치 처리

69개 대안변수 중 `PET_GD_woe`, `APP_GD_woe`, `GOLF_GD_woe`, `TRAVEL_GD_woe` 4개는
신용이력 보유군의 TARGET을 보고 계산하는 값인데, `통신카드CB_씬파일러.csv`에는 신용이력
보유군이 없어(전원 씬파일러) 이 파일만으로는 재계산할 수 없다. track_b_11 결과 기준 이
4개의 SHAP 기여도가 전체의 3~3.3%로 낮은 편이라, **이번 재검증에서는 4개 모두 중립값
0으로 채워서 스코어링**한다(WOE=0은 "정보 없음"을 뜻하는 값이라 임의 추정보다 안전).
신용이력 보유군까지 포함된 원본 파일을 구하면 이 부분만 교체하면 된다.

## 검증 대상 및 계산 항목

- **검증 대상**: `track_b_exit_cohort.csv` 전체 2,931명(실제 연체 39명) — 오염 정제는 이미 끝난 상태
- **기존 신용정보 26개 모델**: track_b_08의 Model A(34개) 튜닝 파라미터 재사용, 100% 원본값이라 근사 없음
- **대안변수 66개 모델**: 69개 중 AGE·JB_TP·HOME_ADM 제외, WOE 4개는 0으로 근사
- **평가**: AUC / PR-AUC / 부트스트랩 95% CI(2,000회) / 순열검정(1,000회)


## 0. 환경 설정

In [7]:
import warnings
warnings.filterwarnings('ignore')

import json
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score
import xgboost as xgb

RANDOM_STATE = 42
handoff_path = r'C:\Users\tehun\Desktop\multicamp\project\creditscore\cardCB'  # track_b_07~10과 동일 폴더
scan_path = r'C:\Users\tehun\Desktop\multicamp\project\data\통신카드CB_씬파일러.csv'  # 실제 경로로 확인된 파일


## 1. 정제 코호트(exit_cohort) 로드

CUST_ID, y_true 두 컬럼만 있음(2,931명, 양성 39명, 오염 정제 이미 완료).

In [8]:
exit_cohort = pd.read_csv(f'{handoff_path}/track_b_exit_cohort.csv')

print(f"정제 코호트: {len(exit_cohort)}명 (기대값: 2,931명)")
print(f"실제 연체(양성): {exit_cohort['y_true'].sum()}명 (기대값: 39명)")

if len(exit_cohort) != 2931 or exit_cohort['y_true'].sum() != 39:
    print("[주의] track_b_11 문서 기준값과 다름 — 파일 버전을 다시 확인할 것")


정제 코호트: 2931명 (기대값: 2,931명)
실제 연체(양성): 39명 (기대값: 39명)


## 2. 기존 신용정보 26개 모델(score_26)

track_b_08 Model A(26개+결측플래그 8개=34개) 최적 파라미터로 신용이력 보유군 285,890명을
학습하고, 이탈 코호트의 202103(씬파일러 시절) 원본 값에 적용한다. 26개 전부 원본 수치라
WOE 근사 문제가 없다.

In [9]:
# --- 학습: 신용이력 보유군 285,890명 (track_b_08과 동일) ---
df_h4 = pd.read_csv(f'{handoff_path}/track_b_traditional_credit_features_for_H4comparison.csv')
traditional_cols = [c for c in df_h4.columns if c not in ['CUST_ID', 'TARGET']]

X_A_full = df_h4[traditional_cols]
y_A_full = df_h4['TARGET']

with open(f'{handoff_path}/track_b_h4_best_params.json', encoding='utf-8') as f:
    h4_params = json.load(f)
params_A = h4_params['model_A_26vars']

model_26 = xgb.XGBClassifier(
    **params_A,
    scale_pos_weight=(y_A_full == 0).sum() / (y_A_full == 1).sum(),
    eval_metric='aucpr', random_state=RANDOM_STATE, n_jobs=-1, tree_method='hist',
)
model_26.fit(X_A_full, y_A_full)
print("기존 신용정보 26개 모델(신용이력보유군 285,890명 학습) 완료")

# --- 이탈 코호트의 202103 원본 값 로드 ---
raw_fields_26 = [c for c in traditional_cols if not c.endswith('_was_missing')]  # 26개 원본 필드
flag_cols_26 = [c for c in traditional_cols if c.endswith('_was_missing')]       # 결측 플래그(8개)

exit_ids = set(exit_cohort['CUST_ID'])
usecols_26 = ['CUST_ID', 'BASE_YM'] + raw_fields_26

chunks = []
for chunk in pd.read_csv(scan_path, usecols=usecols_26, chunksize=100000):
    sub = chunk[(chunk['BASE_YM'] == 202103) & (chunk['CUST_ID'].isin(exit_ids))]
    if len(sub):
        chunks.append(sub)
exit_raw_26 = pd.concat(chunks, ignore_index=True)
print(f"이탈 코호트 202103 원본값 로드: {len(exit_raw_26)}명 (기대값 2,931명)")

# 결측 플래그 생성 (fillna 전에 먼저) 후 원본 0으로 채움 — df_h4와 동일한 처리 순서
for flag_col in flag_cols_26:
    source_col = flag_col[:-len('_was_missing')]
    exit_raw_26[flag_col] = exit_raw_26[source_col].isnull().astype(int)
exit_raw_26[raw_fields_26] = exit_raw_26[raw_fields_26].fillna(0)

exit_x26 = exit_cohort[['CUST_ID']].merge(exit_raw_26, on='CUST_ID', how='left')
if exit_x26[traditional_cols].isnull().any().any():
    n_missing_id = exit_x26[traditional_cols].isnull().any(axis=1).sum()
    print(f"[주의] 이탈 코호트 중 {n_missing_id}명이 원본 파일에서 매칭 안 됨 — CUST_ID 확인 필요")

score_26 = model_26.predict_proba(exit_x26[traditional_cols])[:, 1]
print(f"score_26 산출 완료: {len(score_26)}명")


기존 신용정보 26개 모델(신용이력보유군 285,890명 학습) 완료
이탈 코호트 202103 원본값 로드: 2931명 (기대값 2,931명)
score_26 산출 완료: 2931명


## 3. 대안변수 66개 모델(score_66)

69개 중 `AGE`, `JB_TP`, `HOME_ADM` 제외한 66개로 학습(트리 파라미터는 track_b_08 Model B
재사용). 스코어링 시 `PET_GD_woe`/`APP_GD_woe`/`GOLF_GD_woe`/`TRAVEL_GD_woe` 4개는
0(중립)으로 근사한다.

In [10]:
# --- 학습: 신용이력 보유군 285,890명 (66개, 인구변수 제외) ---
df_train = pd.read_csv(f'{handoff_path}/track_b_features_train_v2.csv')
alt_cols_all = [c for c in df_train.columns if c not in ['CUST_ID', 'TARGET']]

DEMO_EXCLUDE = ['AGE', 'JB_TP', 'HOME_ADM']
found_demo_cols = [c for c in alt_cols_all if any(kw == c.upper() for kw in DEMO_EXCLUDE)]
print(f"제외 대상으로 탐지된 컬럼: {found_demo_cols}")

alt_cols_66 = [c for c in alt_cols_all if c not in found_demo_cols]
print(f"최종 대안변수 개수: {len(alt_cols_66)}개 (기대값 66개)")

X_B_full = df_train[alt_cols_66]
y_B_full = df_train['TARGET']

remaining_categorical = X_B_full.select_dtypes(include='object').columns.tolist()
if remaining_categorical:
    print(f"잔여 범주형 컬럼 발견, 원-핫 인코딩 적용: {remaining_categorical}")
    X_B_full = pd.get_dummies(X_B_full, columns=remaining_categorical, prefix=remaining_categorical)
final_66_cols = X_B_full.columns.tolist()

params_B = h4_params['model_B_26plus69vars']  # 트리 구조 파라미터만 재사용 (피처수 무관)
model_66 = xgb.XGBClassifier(
    **params_B,
    scale_pos_weight=(y_B_full == 0).sum() / (y_B_full == 1).sum(),
    eval_metric='aucpr', random_state=RANDOM_STATE, n_jobs=-1, tree_method='hist',
)
model_66.fit(X_B_full, y_B_full)
print("대안변수 66개 모델(신용이력보유군 285,890명 학습) 완료")

# --- 이탈 코호트의 202103 원본 값으로 66개 피처 재구성 ---
WOE_COLS = [c for c in final_66_cols if c.endswith('_woe')]                     # PET/APP/GOLF/TRAVEL_GD_woe (0 근사)
HAS_GRADE_UNAVAIL = 'grade_unavailable' in final_66_cols
FLAG_COLS_66 = [c for c in final_66_cols if c.endswith('_was_missing')]
NORMAL_COLS_66 = [c for c in final_66_cols
                   if c not in WOE_COLS and c != 'grade_unavailable' and c not in FLAG_COLS_66]

print(f"직접 사용: {len(NORMAL_COLS_66)}개 / 플래그 파생: {len(FLAG_COLS_66)}개 / "
      f"grade_unavailable: {HAS_GRADE_UNAVAIL} / WOE 근사(0): {len(WOE_COLS)}개")

flag_source_cols = [c[:-len('_was_missing')] for c in FLAG_COLS_66]
raw_needed = list(dict.fromkeys(
    NORMAL_COLS_66 + flag_source_cols + (['PET_GD'] if HAS_GRADE_UNAVAIL else [])
))
usecols_66 = ['CUST_ID', 'BASE_YM'] + raw_needed

chunks = []
for chunk in pd.read_csv(scan_path, usecols=usecols_66, chunksize=100000):
    sub = chunk[(chunk['BASE_YM'] == 202103) & (chunk['CUST_ID'].isin(exit_ids))]
    if len(sub):
        chunks.append(sub)
exit_raw_66 = pd.concat(chunks, ignore_index=True)
print(f"이탈 코호트 202103 원본값 로드: {len(exit_raw_66)}명 (기대값 2,931명)")

# 결측 플래그 먼저 생성 (fillna 전)
for flag_col, source_col in zip(FLAG_COLS_66, flag_source_cols):
    exit_raw_66[flag_col] = exit_raw_66[source_col].isnull().astype(int)

# grade_unavailable 파생
if HAS_GRADE_UNAVAIL:
    exit_raw_66['grade_unavailable'] = (exit_raw_66['PET_GD'] == '*').astype(int)

# 원본값 결측 -> 0
exit_raw_66[NORMAL_COLS_66] = exit_raw_66[NORMAL_COLS_66].fillna(0)

# WOE 4개는 중립값 0으로 근사
for c in WOE_COLS:
    exit_raw_66[c] = 0.0

exit_x66 = exit_cohort[['CUST_ID']].merge(exit_raw_66, on='CUST_ID', how='left')
if exit_x66[final_66_cols].isnull().any().any():
    n_missing_id = exit_x66[final_66_cols].isnull().any(axis=1).sum()
    print(f"[주의] 이탈 코호트 중 {n_missing_id}명이 원본 파일에서 매칭 안 됨 — CUST_ID 확인 필요")

score_66 = model_66.predict_proba(exit_x66[final_66_cols])[:, 1]
print(f"score_66 산출 완료: {len(score_66)}명 (WOE 4개 컬럼은 0으로 근사 처리됨)")


제외 대상으로 탐지된 컬럼: ['AGE', 'JB_TP', 'HOME_ADM']
최종 대안변수 개수: 66개 (기대값 66개)
대안변수 66개 모델(신용이력보유군 285,890명 학습) 완료
직접 사용: 49개 / 플래그 파생: 12개 / grade_unavailable: True / WOE 근사(0): 4개
이탈 코호트 202103 원본값 로드: 2931명 (기대값 2,931명)
score_66 산출 완료: 2931명 (WOE 4개 컬럼은 0으로 근사 처리됨)


## 4. 검증 함수 — 부트스트랩 95% CI + 순열검정

- **부트스트랩(2,000회)**: "이 AUC 값이 얼마나 믿을 만한가"를 재표본으로 확인 (신뢰구간)
- **순열검정(1,000회)**: "이 결과가 우연히 나온 게 아닌가"를 라벨 셔플로 확인 (p-value)

In [11]:
def bootstrap_ci(y_true, y_proba, n_boot=2000, seed=42, ci=95):
    rng = np.random.RandomState(seed)
    y_true = np.asarray(y_true); y_proba = np.asarray(y_proba)
    n = len(y_true)
    aucs, praucs = [], []
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        yt, yp = y_true[idx], y_proba[idx]
        if yt.sum() == 0 or yt.sum() == n:
            continue
        aucs.append(roc_auc_score(yt, yp))
        praucs.append(average_precision_score(yt, yp))
    lo, hi = (100 - ci) / 2, 100 - (100 - ci) / 2
    return {
        'auc_mean': float(np.mean(aucs)),
        'auc_ci': (float(np.percentile(aucs, lo)), float(np.percentile(aucs, hi))),
        'prauc_mean': float(np.mean(praucs)),
        'prauc_ci': (float(np.percentile(praucs, lo)), float(np.percentile(praucs, hi))),
        'n_valid_boot': len(aucs),
    }


def permutation_test_auc(y_true, y_proba, n_perm=1000, seed=42):
    rng = np.random.RandomState(seed)
    y_true = np.asarray(y_true)
    observed = roc_auc_score(y_true, y_proba)
    count_ge = 0
    n_valid = 0
    for _ in range(n_perm):
        y_shuffled = rng.permutation(y_true)
        if y_shuffled.sum() == 0 or y_shuffled.sum() == len(y_shuffled):
            continue
        n_valid += 1
        if roc_auc_score(y_shuffled, y_proba) >= observed:
            count_ge += 1
    p_value = count_ge / n_valid if n_valid else np.nan
    return {'observed_auc': observed, 'count_ge': count_ge, 'n_perm_valid': n_valid, 'p_value': p_value}


## 5. 두 모델 검증 실행

In [12]:
y_exit = exit_cohort['y_true'].values

boot_26 = bootstrap_ci(y_exit, score_26, n_boot=2000, seed=RANDOM_STATE)
perm_26 = permutation_test_auc(y_exit, score_26, n_perm=1000, seed=RANDOM_STATE)

boot_66 = bootstrap_ci(y_exit, score_66, n_boot=2000, seed=RANDOM_STATE)
perm_66 = permutation_test_auc(y_exit, score_66, n_perm=1000, seed=RANDOM_STATE)

def format_chance(perm_result, n_perm=1000):
    p = perm_result['p_value']
    if perm_result['count_ge'] == 0:
        return f"0/{n_perm} (p<{1/n_perm:.3f})"
    return f"{perm_result['count_ge']}/{n_perm} ({p:.1%})"

result_table = pd.DataFrame([
    {
        '모델': '기존 신용정보 26개',
        'AUC': round(perm_26['observed_auc'], 4),
        '신뢰구간(95%)': f"[{boot_26['auc_ci'][0]:.2f}, {boot_26['auc_ci'][1]:.2f}]",
        'CI가_0.5_포함': boot_26['auc_ci'][0] <= 0.5 <= boot_26['auc_ci'][1],
        '우연일_가능성': format_chance(perm_26),
    },
    {
        '모델': '대안변수 66개(WOE 4개는 근사)',
        'AUC': round(perm_66['observed_auc'], 4),
        '신뢰구간(95%)': f"[{boot_66['auc_ci'][0]:.2f}, {boot_66['auc_ci'][1]:.2f}]",
        'CI가_0.5_포함': boot_66['auc_ci'][0] <= 0.5 <= boot_66['auc_ci'][1],
        '우연일_가능성': format_chance(perm_66),
    },
])

print(f"검증 코호트: {len(exit_cohort)}명, 실제 연체 {int(y_exit.sum())}명\n")
result_table


검증 코호트: 2931명, 실제 연체 39명



,모델,AUC,신뢰구간(95%),CI가_0.5_포함,우연일_가능성
0,기존 신용정보 26개,0.5265,"[0.43, 0.62]",True,297/1000 (29.7%)
1,대안변수 66개(WOE 4개는 근사),0.7739,"[0.70, 0.84]",False,0/1000 (p<0.001)


## 6. 판정

- **기존 신용정보 26개**: 신뢰구간이 0.5를 포함하면 "무작위 찍기와 통계적으로 구분 안 됨" → track_b_11 원 결과(CI [0.4582, 0.6512], 113/1000)와 같은 결론인지 확인
- **대안변수 66개**: 신뢰구간이 0.5를 확실히 상회하고 순열검정에서 우연일 가능성이 매우 낮으면 → 핵심 주장이 튜닝된 최종 모델로도 재확정
- WOE 4개를 0으로 근사했으므로, track_b_11 원 결과(0.7164)와 차이가 크게 난다면 이 근사의 영향인지 튜닝 자체의 영향인지 구분이 필요 — 신용이력 보유군 포함 원본 파일을 구하면 재검증 권장

## 7. 결과 저장

In [13]:
result_table.to_csv(f'{handoff_path}/track_b_posthoc_validation_final_model.csv', index=False, encoding='utf-8-sig')

summary = {
    'cohort_size': int(len(exit_cohort)),
    'cohort_positive': int(y_exit.sum()),
    'woe_approximation_note': 'PET_GD_woe/APP_GD_woe/GOLF_GD_woe/TRAVEL_GD_woe 4개는 0(중립)으로 근사',
    'score_26': {
        'auc': perm_26['observed_auc'], 'auc_ci': boot_26['auc_ci'],
        'prauc': boot_26['prauc_mean'], 'prauc_ci': boot_26['prauc_ci'],
        'permutation_p_value': perm_26['p_value'],
    },
    'score_66': {
        'auc': perm_66['observed_auc'], 'auc_ci': boot_66['auc_ci'],
        'prauc': boot_66['prauc_mean'], 'prauc_ci': boot_66['prauc_ci'],
        'permutation_p_value': perm_66['p_value'],
    },
    'track_b_11_reference(고정파라미터, 5필드 확장 정제 코호트)': {
        'score_26_auc': 0.5570, 'score_26_ci': [0.4582, 0.6512],
        'score_66_auc': 0.7164, 'score_66_ci': [0.6470, 0.7813],
    },
}

with open(f'{handoff_path}/track_b_posthoc_validation_final_model.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2, default=str)

print("저장 완료: track_b_posthoc_validation_final_model.csv, track_b_posthoc_validation_final_model.json")
print(result_table)


저장 완료: track_b_posthoc_validation_final_model.csv, track_b_posthoc_validation_final_model.json
                     모델     AUC     신뢰구간(95%)  CI가_0.5_포함           우연일_가능성
0           기존 신용정보 26개  0.5265  [0.43, 0.62]        True  297/1000 (29.7%)
1  대안변수 66개(WOE 4개는 근사)  0.7739  [0.70, 0.84]       False  0/1000 (p<0.001)
